# Chapter 7 - Distributional Q-learning

### Code for Chapter 7 of _Deep Reinforcement Learning in Action_ 
#### by Alex Zai and Brandon Brown
An implementation of "A Distributional Perspective on Reinforcement Learning" by Bellemare et al 2017

In [ ]:
import numpy as np

probs = np.array([0.6, 0.1, 0.1, 0.1, 0.1])
outcomes = np.array([18, 21, 17, 17, 21])
expected_value = 0.0
for i in range(probs.shape[0]):
    expected_value += probs[i] * outcomes[i]

print(expected_value)

In [ ]:
expected_value = probs @ outcomes
print(expected_value)

In [ ]:
t0 = 18.4
T = lambda: t0 + np.random.randn(1)
T()

##### Listing 7.1 - Setting up a discrete probability distribution in numpy

In [ ]:
import torch
import numpy as np
from matplotlib import pyplot as plt

uniform_dist = lambda n_sup: np.ones(n_sup) / n_sup

vmin, vmax = -10.0, 10.0  # A
nsup = 51  # B
support = np.linspace(vmin, vmax, nsup)  # C
probs = uniform_dist(nsup)
plt.bar(support, probs)  # D

##### Listing 7.2 - Updating a probability distribution

In [ ]:
def update_dist(r, support, probs, lim=(-10.0, 10.0), gamma=0.8):
    """
    This function takes a discrete probability distribution over the range of rewards
    and updates the distribution based on a single new reward observation.
    `r` is a float indicating the observed reward
    `support`, is a vector of the distribution's support, e.g. uniformly spaced numbers from -10 to +10
    `probs`, is the vector of probabilities over `support`
    `lim`, tuple of floats, are the lower and upper limits of the support
    `gamma`, float, is the discount factor
    """
    nsup = len(support)
    vmin, vmax = lim[0], lim[1]
    dz = (vmax - vmin) / (nsup - 1.0)  # A
    bj = np.round((r - vmin) / dz)  # B
    bj = int(np.clip(bj, 0, nsup - 1))  # C
    m = probs.clone()
    j = 1
    for i in range(bj, 1, -1):  # D
        m[i] += np.power(gamma, j) * m[i - 1]
        j += 1
    j = 1
    for i in range(bj, nsup - 1, 1):  # E
        m[i] += np.power(gamma, j) * m[i + 1]
        j += 1
    m /= m.sum()  # F
    return m

##### Listing 7.3 - Redistributing probability mass after a single observation

Now we simulate observing a sequence of rewards and see how it changes the distribution. You will see little peaks at the observed rewards but it is still fairly uniform. Try changing the `gamma` to see how this affects the distribution updating. A small `gamma` will result in small updates from prior to posterior distribution, whereas a `gamma` closer to `1` will result in large updates.

In [ ]:
ob_reward = -1
Z = torch.from_numpy(probs).float()
support_t = torch.from_numpy(support).float()

fig, ax = plt.subplots(11, 1, figsize=(4, 22))
for i in range(0, 11):
    g = i / 10
    zi = update_dist(ob_reward, support_t, Z, lim=(vmin, vmax), gamma=g)
    ax[i].bar(support, zi.detach().cpu().numpy())
    ax[i].set_title(f"gamma={g:.1f}")
    ax[i].set_xlim(vmin, vmax)

plt.tight_layout()
plt.show()

##### Listing 7.4 - Redistributing probability mass with a sequence of observations

In [ ]:
Z = torch.from_numpy(np.ones(nsup) / nsup).float()
ob_rewards = [10, 10, 10, 0, 1, 0, -10, -10, 10, 10]
for i in range(len(ob_rewards)):
    Z = update_dist(
        ob_rewards[i], torch.from_numpy(support).float(), Z, lim=(vmin, vmax), gamma=0.5
    )
plt.bar(support, Z)

##### Listing 7.5 - Decreased variance with sequence of same reward

In [ ]:
Z = torch.from_numpy(np.ones(nsup) / nsup).float()
ob_rewards = [5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5]
for i in range(len(ob_rewards)):
    Z = update_dist(
        ob_rewards[i], torch.from_numpy(support).float(), Z, lim=(vmin, vmax), gamma=0.7
    )
plt.bar(support, Z)

##### Listing 7.6 - The Dist-DQN

In [ ]:
STATE_INPUT_SIZE = 128


def dist_dqn(x, theta, aspace=3):  # A
    """
    3 layer neural network
    `x` is input vector of dim 128
    `theta` is a parameter vector that will be unpacked into 3 separate layer matrices
        layer1: L1 x X -> 100x128 x 128xB -> 100xB
        layer2: L2 x L1 -> 25x100 x 100xB -> 25xB
        layer3: L3 x L2 -> 3x25x51 x 25xB -> 3x51xB
        where `B` is the batch size dimension

    Returns a Batch Sizex A x 51 tensor where A is the action-space size
    """
    dim0, dim1, dim2, dim3 = STATE_INPUT_SIZE, 100, 25, 51  # B
    t1 = dim0 * dim1
    t2 = dim2 * dim1
    theta1 = theta[0:t1].reshape(dim0, dim1)  # C
    theta2 = theta[t1 : t1 + t2].reshape(dim1, dim2)
    l1 = x @ theta1  # D
    l1 = torch.selu(l1)
    l2 = l1 @ theta2  # E
    l2 = torch.selu(l2)
    l3 = []
    for i in range(aspace):  # F
        step = dim2 * dim3
        theta5_dim = t1 + t2 + i * step
        theta5 = theta[theta5_dim : theta5_dim + step].reshape(dim2, dim3)
        l3_ = l2 @ theta5  # G
        l3.append(l3_)
    l3 = torch.stack(l3, dim=1)  # H
    l3 = torch.nn.functional.softmax(l3, dim=2)
    return l3.squeeze()

##### Listing 7.7 - Computing the target distribution

In [ ]:
def get_target_dist(
    dist_batch, action_batch, reward_batch, support, lim=(-10, 10), gamma=0.8, terminal_batch=None, current_dist_batch=None
):
    """
    Given a mini-batch of Q-value distribution predictions,
    this function produces the target distribution used in the loss
    function to update the DQN

    `dist_batch`: Ax51xB where A is the action space size and B is batch size
    `action_batch`: dim=B vector of integers in {0,1,2} of actions
    `reward_batch`: dim=B vector of floats
    `lim`:2-tuple, indicates the lower and upper bound of the support
    `gamma`, float in [0,1], the discount factor
    """
    # The toy listing treats every reward other than -1 as an immediate outcome.
    # Environment callers supply explicit outcome flags instead of inferring from sign.
    if terminal_batch is None:
        terminal_batch = reward_batch != -1
    nsup = support.shape[0]
    vmin, vmax = lim[0], lim[1]
    dz = (vmax - vmin) / (nsup - 1.0)
    target_dist_batch = (dist_batch if current_dist_batch is None else current_dist_batch).detach().clone()
    for i in range(dist_batch.shape[0]):  # A
        dist_full = dist_batch[i]
        action = int(action_batch[i].item())
        dist = dist_full[action]
        r = reward_batch[i]
        if terminal_batch[i]:  # B
            target_dist = torch.zeros_like(dist)
            bj = np.round((r - vmin) / dz)
            bj = int(np.clip(bj, 0, nsup - 1))
            target_dist[bj] = 1.0
        else:  # C
            target_dist = update_dist(r, support, dist, lim=lim, gamma=gamma)
        target_dist_batch[i, action, :] = target_dist  # D

    return target_dist_batch

##### Listing 7.8 - The cross-entropy loss function

In [ ]:
def lossfn(x, y):  # A
    """
    Cross-Entropy Loss between prediction distribution `x` and target distribution `y`
    `x`: B x A x 51 tensor where `B` is batch dimension, `A` is action dimension
    `y` is B x A x 51 tensor

    Output
    - `loss`: Float
    """
    loss = torch.Tensor([0.0])
    loss.requires_grad = True
    for i in range(x.shape[0]):  # B
        loss_ = (
            -1 * torch.log(x[i].flatten(start_dim=0).clamp_min(torch.finfo(x.dtype).tiny)) @ y[i].flatten(start_dim=0)
        )  # C
        loss = loss + loss_
    return loss

##### Listing 7.9 - Testing with simulated data

To test the Dist-DQN and other functions, we train the dist-DQN to learn the reward distributions for 2 observed states (randomly created).

In [ ]:
aspace = 3  # A
tot_params = STATE_INPUT_SIZE * 100 + 25 * 100 + aspace * 25 * 51  # B
theta = torch.randn(tot_params) / 10.0  # C
theta.requires_grad = True
theta_2 = theta.detach().clone()  # D
#
vmin, vmax = -10, 10
gamma = 0.9
lr = 0.005
update_rate = 75  # E
support = torch.linspace(-10, 10, 51)
state = torch.randn(2, STATE_INPUT_SIZE) / 10.0  # F
action_batch = torch.Tensor([0, 2])  # G
reward_batch = torch.Tensor([0, 10])  # H
losses = []
pred_batch = dist_dqn(state, theta, aspace=aspace)  # I
target_dist = get_target_dist(
    pred_batch, action_batch, reward_batch, support, lim=(vmin, vmax), gamma=gamma
)  # J

plt.plot(
    (target_dist.flatten(start_dim=1)[0].data.numpy()), color="red", label="target"
)
plt.plot((pred_batch.flatten(start_dim=1)[0].data.numpy()), color="green", label="pred")
plt.legend()

##### Listing 7.10 - Dist-DQN training on synthetic data

In [ ]:
for i in range(1000):
    reward_batch = torch.Tensor([0, 8]) + torch.randn(2) / 10.0  # A
    pred_batch = dist_dqn(state, theta, aspace=aspace)  # B
    pred_batch2 = dist_dqn(state, theta_2, aspace=aspace)  # C
    target_dist = get_target_dist(
        pred_batch2, action_batch, reward_batch, support, lim=(vmin, vmax), gamma=gamma,
        current_dist_batch=pred_batch
    )  # D
    loss = lossfn(pred_batch, target_dist.detach())  # E
    losses.append(loss.item())
    theta.grad = None
    loss.backward()
    # Gradient Descent
    with torch.no_grad():
        theta -= lr * theta.grad
    theta.requires_grad = True

    if i % update_rate == 0:  # F
        theta_2 = theta.detach().clone()

fig, ax = plt.subplots(2, 1, figsize=(6, 10))
ax[0].plot(
    (target_dist.flatten(start_dim=1)[0].data.numpy()), color="red", label="target"
)
ax[0].plot(
    (pred_batch.flatten(start_dim=1)[0].data.numpy()), color="green", label="pred"
)
ax[0].legend()
ax[1].plot(losses)

##### Listing 7.11 - Visualizing the learned action-value distributions

In [ ]:
tpred = pred_batch
cs = ["gray", "green", "red"]
num_batch = 2
fig, ax = plt.subplots(nrows=num_batch, ncols=aspace)
fig.set_size_inches((15, 5))

for j in range(num_batch):  # A
    for i in range(tpred.shape[1]):  # B
        ax[j, i].bar(
            support.data.numpy(),
            tpred[j, i, :].data.numpy(),
            label=f"Action {i}",
            alpha=0.9,
            color=cs[i],
        )

##### Listing 7.12 - Preprocessing states and selecting actions

In [ ]:
def preproc_state(state):
    """
    Takes numpy array from env.reset or env.step
    and converts to PyTorch Tensor, adds batch dimension and normalizes

    Output:
    - `p_state`: PyTorch tensor of dimensions 1x128
    """
    p_state = torch.from_numpy(state).unsqueeze(dim=0).float()
    p_state = torch.nn.functional.normalize(p_state, dim=1)  # A
    return p_state


def get_action(dist, support):
    """
    This function returns an integer action in [0,1,2]
    `dist` input is a Ax51xB discrete distribution over Q-values for each action
    where `A` is the action-space size, and `B` is the batch dimension.
    Get expectations w.r.t. to each action, take action w/ highest q-value

    Output:
    - `actions`: vector of integers in {0,1,2}, dimension dist.shape[0] (batch size)
    """
    actions = []
    for b in range(dist.shape[0]):  # B
        expectations = [support @ dist[b, a, :] for a in range(dist.shape[1])]  # C
        action = int(np.argmax(expectations))  # D
        actions.append(action)
    actions = torch.Tensor(actions).int()
    return actions

##### Listing 7.13 - Dist-DQN plays Freeway, preliminaries

Create the gym environment for Atari Freeway.
To simplify the algorithm and focus on the topics of interest, we use the ram version of Freeway rather than the raw pixels. In the ram version, the environment's states are 128-element vectors as opposed to RGB frames. This allows us to use a much simpler (i.e. non-convolutional, fewer layers) neural network so we can see the results of training faster.

In [ ]:
import ale_py
import gymnasium as gym
from collections import deque

gym.register_envs(ale_py)

env = gym.make("ALE/Freeway-v5", obs_type="ram", frameskip=(2, 5), repeat_action_probability=0.25)

action_names = env.unwrapped.get_action_meanings()
aspace = len(action_names)
print("Actions:", action_names)

vmin, vmax = -10, 10
replay_size = 200
batch_size = 50
nsup = 51
dz = (vmax - vmin) / (nsup - 1)
support = torch.linspace(vmin, vmax, nsup)

replay = deque(maxlen=replay_size)  # A
lr = 0.01  # B
gamma = 0.1  # C
epochs = 10000
eps = 0.2  # D starting epsilon for epsilon-greedy policy
eps_min = 0.05  # E ending epsilon
# prioritized-replay; duplicate high-value experiences in the replay
priority_level = 5  # F
update_freq = 25  # G

# Initialize DQN parameter vector
tot_params = STATE_INPUT_SIZE * 100 + 25 * 100 + aspace * 25 * 51  # H
theta = torch.randn(tot_params) / 10.0  # I
theta.requires_grad = True
theta_2 = theta.detach().clone()  # J

losses = []
cum_rewards = []  # K
renders = []
state = preproc_state(env.reset()[0])
print(state.shape)

##### Listing 7.14 - The main training loop

In [ ]:
from random import shuffle

ep_len = 0
ep_lens = []
for i in range(epochs):
    pred = dist_dqn(state, theta, aspace=aspace)
    if i < replay_size or np.random.rand(1) < eps:  # A
        action = np.random.randint(aspace)
    else:
        action = get_action(pred.unsqueeze(dim=0).detach(), support).item()
    state2, reward, terminated, truncated, _ = env.step(action)  # B
    done = terminated or truncated
    state2 = preproc_state(state2)
    if reward == 1:  # C
        print(f"Won at i={i}, steps={ep_len}")
        reward = 10
        ep_lens.append(ep_len)
        ep_len = 0
        cum_rewards.append(1)
    elif terminated:  # D
        print(f"Terminated at i={i}, steps={ep_len}")
        reward = -10 if ep_len > 1000 else -1
        ep_lens.append(ep_len)
        ep_len = 0
    else:  # E
        reward = -1
        ep_len += 1
    exp = (state, action, reward, state2, terminated or reward == 10)  # F
    replay.append(exp)  # G

    if reward == 10:  # H
        for e in range(priority_level):
            replay.append(exp)

    shuffle(replay)
    state = state2

    if len(replay) == replay_size:  # I
        indx = np.random.randint(low=0, high=len(replay), size=batch_size)
        exps = [replay[j] for j in indx]
        state_batch = torch.stack([ex[0] for ex in exps], dim=1).squeeze()
        action_batch = torch.Tensor([ex[1] for ex in exps])
        reward_batch = torch.Tensor([ex[2] for ex in exps])
        state2_batch = torch.stack([ex[3] for ex in exps], dim=1).squeeze()
        pred_batch = dist_dqn(state_batch.detach(), theta, aspace=aspace)
        pred2_batch = dist_dqn(state2_batch.detach(), theta_2, aspace=aspace)
        target_dist = get_target_dist(
            pred2_batch,
            action_batch,
            reward_batch,
            support,
            lim=(vmin, vmax),
            gamma=gamma,
            terminal_batch=torch.tensor([ex[4] for ex in exps]),
            current_dist_batch=pred_batch,
        )
        loss = lossfn(pred_batch, target_dist.detach())
        losses.append(loss.item())
        theta.grad = None
        loss.backward()
        with torch.no_grad():  # J
            theta -= lr * theta.grad
        theta.requires_grad = True

    if i % update_freq == 0:  # K
        theta_2 = theta.detach().clone()

    if i > 100 and eps > eps_min:  # L
        dec = 1.0 / np.log2(i)
        dec /= 1e3
        eps -= dec

    if done:  # M
        state = preproc_state(env.reset()[0])

In [ ]:
plt.plot(losses)

In [ ]:
# first completion is pure luck
plt.plot(ep_lens[1:])

Display the reward distributions for each action for a random sample of 5 experiences in the replay buffer:

In [ ]:
tpred = pred_batch
cs = ["yellow", "green", "red"]
num_batch = 5
batch_ind = np.random.randint(0, tpred.shape[0], num_batch)

fig, ax = plt.subplots(nrows=num_batch, ncols=aspace)
fig.set_size_inches((15, 15))
for j_ in range(num_batch):  # loop through 5 first experiences in batch
    j = batch_ind[j_]
    for i in range(tpred.shape[1]):  # loop through actions
        ax[j_, i].bar(
            support.data.numpy(),
            tpred[j, i, :].data.numpy(),
            label="Action {}".format(i),
            alpha=0.9,
            color=cs[i],
        )
        ev = support.data.numpy() @ tpred[j, i, :].data.numpy()
        ax[j_, i].set_title("$\\mathbb{E} = $" + str(ev))
fig.tight_layout()

In [ ]:
import time
from IPython.display import clear_output

env = gym.make(
    "ALE/Freeway-v5",
    render_mode="rgb_array",
    obs_type="ram", frameskip=(2, 5), repeat_action_probability=0.25,
)
state, _ = env.reset()
state = preproc_state(state)

stop_at = 5
cum_rewards = 0
done = False
steps = 0
while not done:
    pred = dist_dqn(state, theta, aspace=aspace)
    action = get_action(pred.unsqueeze(dim=0).detach(), support).item()

    state, reward, terminated, truncated, _ = env.step(action)
    state = preproc_state(state)
    if reward == 1:
        steps = 0

    done = terminated or truncated or cum_rewards == stop_at
    cum_rewards += reward
    steps += 1

    # Draw periodically without changing the policy or environment steps.
    if steps % 10 and not done:
        continue

    # Render environment
    fig, ax = plt.subplots(1, 2, figsize=(10, 6))
    frame = env.render()
    ax[0].imshow(frame)
    ax[0].set_title(f"Score: {cum_rewards}, Steps: {steps}")
    ax[0].axis("off")

    # Render distrubution
    cs = ["yellow", "green", "red"]
    ax[1].set_ylim(0, 0.5)
    for i in range(len(action_names)):
        ax[1].bar(
            support.data.numpy(),
            pred[i, :].data.numpy(),
            label=action_names[i],
            alpha=0.9,
            color=cs[i],
        )

    ax[1].legend(loc="upper right", prop={"size": 10})

    clear_output(wait=True)
    plt.show()
    plt.close(fig)
    time.sleep(0.1)

env.close()